# Empirical study v2: multi-seed, CLIP + DINOv2, Fashion Product Images (small)
GPU runtime. Expect ~40-60 min for 5 seeds x 2 encoders (each run re-extracts features for its own split).

In [ ]:
!rm -rf semantic-chromatic-retrieval && git clone https://github.com/MokshdaSharma/semantic-chromatic-retrieval.git
%cd semantic-chromatic-retrieval
!pip install -q -r requirements.txt

In [ ]:
# More data than v1 (v1 kept ~9.4k images after the class filter). Raise/lower --n as you like.
!python scripts/prepare_fashion_small.py --out /content/fashion_small --n 30000

In [ ]:
import subprocess, sys
SEEDS = [0, 1, 2, 3, 4]
for s in SEEDS:
    for e in ['clip', 'dinov2']:
        out = f'results/v2/fs_{e}_s{s}'
        r = subprocess.run([sys.executable, '-m', 'cr.run', '--dataset', 'fashion_small', '--root', '/content/fashion_small', '--encoder', e, '--seed', str(s), '--out', out])
        print(out, 'exit', r.returncode)

In [ ]:
!python -m cr.aggregate --runs results/v2/fs_clip_s* results/v2/fs_dinov2_s* --out results/v2_tables

In [ ]:
import pandas as pd; pd.set_option('display.width', 250)
print(pd.read_csv('results/v2_tables/table_main.csv').to_string(index=False))
print(pd.read_csv('results/v2_tables/table_paired.csv').to_string(index=False))
print(pd.read_csv('results/v2_tables/table_metric_agreement.csv').to_string(index=False))

In [ ]:
# Download results (small CSV/JSON/PNG only; feature caches excluded)
!zip -qr results_v2.zip results/v2 results/v2_tables -x '*/cache/*'
from google.colab import files; files.download('results_v2.zip')